<a href="https://colab.research.google.com/github/KalitonOliveira001/KalitonOliveira001/blob/main/Projeto_Agregra%C3%A7%C3%B5es_iynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
#Importando as bibliotecas necessárias
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg, max, min, count, variance, first, last, sum as sum_
from pyspark.sql import SparkSession
from pyspark.sql.functions import round

In [ ]:
#Uploaded dos arqivos
from google.colab import files
uploaded = files.upload()

Saving videos-preparados.snappy.parquet to videos-preparados.snappy (1).parquet


In [ ]:
# Criando uma sessão do Spark
spark = SparkSession.builder \
    .appName("Agregações em Dados") \
    .getOrCreate()

In [ ]:
# Lendo o arquivo Parquet
df_video = spark.read.parquet('videos-preparados.snappy.parquet')

In [ ]:
# 1. Calcule a quantidade de registros para cada valor exclusivo da coluna "Palavra-chave"
df_keyword_count = df_video.groupBy("keyword").agg(count("*").alias("Quantidade Registros"))
df_keyword_count.show()


+----------------+--------------------+
|         keyword|Quantidade Registros|
+----------------+--------------------+
|computer science|                  48|
|            lofi|                  40|
|         finance|                  39|
|             cnn|                  50|
|           apple|                  42|
|            news|                  39|
|         mukbang|                  45|
|       education|                  24|
|       interview|                  50|
|          crypto|                  50|
|   mathchemistry|                  15|
|            food|                  48|
|    data science|                  50|
|        trolling|                  50|
|        tutorial|                  50|
|      literature|                  46|
|             sat|                  49|
|         history|                  49|
|           cubes|                  49|
|           music|                  46|
+----------------+--------------------+
only showing top 20 rows



In [ ]:
# 2. Calcule a média da coluna "Interaction" para cada valor único da coluna 'Keyword'
df_interaction_avg = df_video.groupBy("Keyword").agg(avg("Interaction").alias("Média Interação"))
df_interaction_avg.show()


+----------------+--------------------+
|         Keyword|     Média Interação|
+----------------+--------------------+
|computer science|  1226793.0208333333|
|            lofi|         4167085.875|
|         finance|   708542.9487179487|
|             cnn|           570650.86|
|           apple|1.0873628214285715E7|
|            news|  251688.71794871794|
|         mukbang|1.1053630377777778E7|
|       education|         2750838.625|
|       interview|          3044867.04|
|          crypto|            413676.2|
|   mathchemistry|  3427342.7333333334|
|            food|   5352944.104166667|
|    data science|           562465.28|
|        trolling|          1484584.88|
|        tutorial|           6936688.3|
|      literature|            881726.5|
|             sat|           1098927.0|
|         history| 1.565269257142857E7|
|           cubes|1.5043961224489795E7|
|           music|2.9691370304347824E7|
+----------------+--------------------+
only showing top 20 rows



In [ ]:
# 3. Valor máximo da coluna "Interaction" para cada valor único da coluna "Keyword",
# nomeando como 'Rank Interactions' e ordenando decrescente
df_rank_interactions = df_video.groupBy("Keyword") \
    .agg(max("Interaction").alias("Rank Interactions")) \
    .orderBy(col("Rank Interactions").desc())
df_rank_interactions.show()


+--------+-----------------+
| Keyword|Rank Interactions|
+--------+-----------------+
| animals|       1593623628|
|   music|        922551152|
|     bed|        532691631|
| history|        440187490|
|   apple|        429916936|
| mrbeast|        300397699|
|  google|        239385460|
|business|        210025196|
|   cubes|        170925917|
|  sports|        106924567|
| mukbang|         87433858|
|    lofi|         86445177|
|tutorial|         69616442|
|  movies|         65253870|
|  marvel|         56247330|
|  how-to|         53053975|
|    food|         48754479|
| physics|         43463298|
|    asmr|         34411125|
|nintendo|         32268486|
+--------+-----------------+
only showing top 20 rows



In [ ]:
# 4. Média e variância da coluna 'Views' para cada valor único da coluna "Keyword"
df_views_stats = df_video.groupBy("Keyword").agg(
    avg("Views").alias("Média Visualizações"),
    variance("Views").alias("Variância Visualizações")
)
df_views_stats.show()


+----------------+--------------------+-----------------------+
|         Keyword| Média Visualizações|Variância Visualizações|
+----------------+--------------------+-----------------------+
|computer science|  1191958.7083333333|    2.81219868165102E12|
|            lofi|           4089363.0|   1.846209641476677...|
|         finance|   694223.4358974359|   3.304483175097042...|
|             cnn|           554240.38|   1.563423618468118...|
|           apple|1.0746930452380951E7|   4.299927977442589E15|
|            news|   247492.1794871795|   1.067512576672564...|
|         mukbang|1.0904772355555555E7|   5.586073238973179...|
|       education|  2684432.8333333335|   1.833572249339214...|
|       interview|          2966111.86|   1.819220996034335E13|
|          crypto|           404608.22|   3.513691634369074E12|
|   mathchemistry|  3328125.2666666666|   2.491467065256849...|
|            food|          5252406.25|   7.326374128154842E13|
|    data science|           544771.98| 

In [ ]:
# 5. Média, valor mínimo e máximo de 'Views' para cada "Keyword", sem casas decimais
df_views_summary = df_video.groupBy("Keyword").agg(
    round(avg("Views")).cast("int").alias("Média Visualizações"),
    min("Views").alias("Mínimo Visualizações"),
    max("Views").alias("Máximo Visualizações")
)
df_views_summary.show()


+----------------+-------------------+--------------------+--------------------+
|         Keyword|Média Visualizações|Mínimo Visualizações|Máximo Visualizações|
+----------------+-------------------+--------------------+--------------------+
|computer science|            1191959|               16115|             7004107|
|            lofi|            4089363|                6817|            84747957|
|         finance|             694223|                1195|             9450554|
|             cnn|             554240|               51269|             1889320|
|           apple|           10746930|                1954|           425478119|
|            news|             247492|               10642|             1465011|
|         mukbang|           10904772|                3618|            86169225|
|       education|            2684433|                6611|            17103736|
|       interview|            2966112|                2587|            22529756|
|          crypto|          

In [ ]:
# 6. Primeiro e último 'Published At' para cada valor exclusivo da coluna "Keyword"
df_published_dates = df_video.groupBy("Keyword").agg(
    first("Published At").alias("Primeiro Published At"),
    last("Published At").alias("Último Published At")
)
df_published_dates.show()


+----------------+---------------------+-------------------+
|         Keyword|Primeiro Published At|Último Published At|
+----------------+---------------------+-------------------+
|computer science|           2022-02-08|         2020-09-08|
|            lofi|           2022-06-07|         2020-07-19|
|         finance|           2020-09-23|         2017-12-31|
|             cnn|           2022-08-17|         2022-08-13|
|           apple|           2022-08-22|         2022-08-02|
|            news|           2022-08-22|         2022-08-23|
|         mukbang|           2020-04-18|         2022-08-24|
|       education|           2015-02-06|         2010-10-14|
|       interview|           2021-08-03|         2018-10-05|
|          crypto|           2022-08-23|         2022-08-22|
|   mathchemistry|           2020-08-11|         2019-10-04|
|            food|           2022-07-17|         2022-08-20|
|    data science|           2019-08-18|         2021-08-06|
|        trolling|      

In [ ]:
# 7. Contar todos os 'Title' e títulos únicos; verificar diferença
total_titles = df_video.select("Title").count()
unique_titles = df_video.select("Title").distinct().count()
print(f"Total de títulos: {total_titles}, Total de títulos únicos: {unique_titles}")


Total de títulos: 1869, Total de títulos únicos: 1854


In [ ]:
# 8. Quantidade de registros ordenados por ano em ordem ascendente
df_year_count = df_video.groupBy("Year").agg(count("*").alias("Quantidade Registros")).orderBy("Year")
df_year_count.show()  # Adicione os parênteses aqui


+----+--------------------+
|Year|Quantidade Registros|
+----+--------------------+
|2007|                   2|
|2008|                   1|
|2009|                   9|
|2010|                   6|
|2011|                   4|
|2012|                  12|
|2013|                   6|
|2014|                  10|
|2015|                  15|
|2016|                  34|
|2017|                  47|
|2018|                  57|
|2019|                  86|
|2020|                 158|
|2021|                 229|
|2022|                1193|
+----+--------------------+



In [ ]:
# 9. Quantidade de registros ordenados por ano e mês em ordem ascendente
df_month_count = df_video.groupBy("Year", "Month").agg(count("*").alias("Quantidade Registros")).orderBy("Year", "Month")
df_month_count.show()

+----+-----+--------------------+
|Year|Month|Quantidade Registros|
+----+-----+--------------------+
|2007|    7|                   1|
|2007|   12|                   1|
|2008|    7|                   1|
|2009|    2|                   2|
|2009|    6|                   2|
|2009|    7|                   1|
|2009|    8|                   1|
|2009|   10|                   1|
|2009|   12|                   2|
|2010|    3|                   1|
|2010|    5|                   2|
|2010|    6|                   1|
|2010|    9|                   1|
|2010|   10|                   1|
|2011|    2|                   1|
|2011|    5|                   1|
|2011|    9|                   1|
|2011|   10|                   1|
|2012|    1|                   1|
|2012|    2|                   3|
+----+-----+--------------------+
only showing top 20 rows



In [ ]:
# 10. Média acumulativa de 'Likes' para cada 'Keyword' ao longo dos anos
window_spec = Window.partitionBy("Keyword").orderBy("Year").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_likes_cumulative = df_video.withColumn("Likes Acumulativos", sum_("Likes").over(window_spec))
df_likes_cumulative.select("Keyword", "Year", "Likes", "Likes Acumulativos").orderBy("Keyword", "Year").show()


+-------+----+--------+------------------+
|Keyword|Year|   Likes|Likes Acumulativos|
+-------+----+--------+------------------+
|animals|2009| 1357197|           1357197|
|animals|2010|   68133|           1425330|
|animals|2010|  338601|           1763931|
|animals|2013|11025176|          12789107|
|animals|2014| 1019385|          13808492|
|animals|2014| 5743875|          19552367|
|animals|2019| 1103713|          20656080|
|animals|2020|   74617|          20730697|
|animals|2020|    9313|          20740010|
|animals|2020|  164337|          20904347|
|animals|2020|   94089|          20998436|
|animals|2020|   21946|          21020382|
|animals|2020|   28863|          21049245|
|animals|2020|   73362|          21122607|
|animals|2020|  282754|          21405361|
|animals|2020| 6177588|          27582949|
|animals|2021|   11323|          27594272|
|animals|2021|   27102|          27621374|
|animals|2021|  120500|          27741874|
|animals|2021|  639942|          28381816|
+-------+--

In [ ]:
# Encerre a sessão Spark se desejar
spark.stop()

